In [ ]:
# %cd /opt/app-root/src/llm/Neeko-main

In [2]:
!pip install -r requirements.txt
!pip install jsonargparse
!pip install tf-keras

^C
  Using cached jsonargparse-4.43.0-py3-none-any.whl.metadata (12 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\pivar\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
!pip install transformers sentence-transformers
from huggingface_hub import snapshot_download

snapshot_download(
    local_dir_use_symlinks=True,
    repo_type="dataset",
    repo_id="fnlp/character-llm-data",
    local_dir="./data/dataset")

^C


ModuleNotFoundError: No module named 'huggingface_hub'

In [ ]:
!python shuffle_data.py \
    --data_dir ./data/dataset \
    --out_path ./data/shuffle.jsonl

In [ ]:
!python embd_roles.py \
    --encoder_path "google-bert/bert-large-uncased" \
    --seed_data_path ./data/seed_data \
    --save_path ./data/embed

In [1]:
!huggingface-cli login --token ""

Could not connect to 127.0.0.1: 57516
Traceback (most recent call last):
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_comm.py", line 468, in start_client
    s.connect((host, port))
    ~~~~~~~~~^^^^^^^^^^^^^^
ConnectionRefusedError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение
Traceback (most recent call last):
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\jupyter_debug\pydev_jupyter_utils.py", line 83, in attach_to_debugger
    debugger.connect(pydev_localhost.get_localhost(), debugger_port)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\pydevd.py", line 704, in connect
    s = start_client(host, port)
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_comm.py", line 468, in

In [1]:
!echo "3" | python train.py \
    --model_name_or_path "meta-llama/Llama-3.2-1B"  \
    --use_fast_tokenizer \
    --data_path ./data/shuffle.jsonl \
    --embds_dir ./data/embed \
    --do_train \
    --finetuning_type moelora \
    --output_dir ./data/train_output/ \
    --max_source_length 4096 \
    --overwrite_cache \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --save_steps 1000 \
    --learning_rate 2e-4 \
    --num_train_epochs 0.1 \
    --lora_rank 32 \
    --num_moe 8 \
    --gating Dense \
    --fp16 \
    --remove_unused_columns False \
    --dataset character-llm

Could not connect to 127.0.0.1: 49671
Traceback (most recent call last):
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_comm.py", line 468, in start_client
    s.connect((host, port))
    ~~~~~~~~~^^^^^^^^^^^^^^
ConnectionRefusedError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение
Traceback (most recent call last):
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\jupyter_debug\pydev_jupyter_utils.py", line 83, in attach_to_debugger
    debugger.connect(pydev_localhost.get_localhost(), debugger_port)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\pydevd.py", line 704, in connect
    s = start_client(host, port)
  File "C:\Users\pivar\AppData\Local\Programs\PyCharm\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_comm.py", line 468, in

In [ ]:

import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the base model name
with open("./data/train_output/adapter_config.json", "r") as f:
    adapter_config = json.load(f)

base_model_path = adapter_config["base_model_name_or_path"]

# Load the base model
model = AutoModelForCausalLM.from_pretrained(base_model_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Load the LoRA adapter to base model
adapter_weights = torch.load("./data/train_output/adapter_model.bin")
model.load_state_dict(adapter_weights, strict=False)


def ask_model(prompt):
    system_prompt = """
    I want you to act like {character}. I want you to respond and answer like {character}, using the tone, manner and vocabulary {character} would use. You must know all of the knowledge of {character}.

    The status of you is as follows:
    Location: {loc_time}
    Status: {status}

    The interactions are as follows:
    """
    full_prompt = system_prompt + "\nUser: " + prompt + ":"
    # Tokenize input
    inputs = tokenizer(full_prompt, return_tensors="pt")

    # Generate output
    outputs = model.generate(
        inputs["input_ids"],
        max_length=200,
        num_return_sequences=1,
        temperature=0.7
    )

    # Decode the output
    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return generated_text


while True:
    input_text = input("Ask your question: ")
    response = ask_model(input_text)
    print("Model's response:", response)
    cont = input("Continue?(Y/n)")
    if cont == 'n':
        break
